## Разработка модели оценки стоимости автомобилей на основе данных Дром.ру
# Задача:
Обучим модель для предсказания стоимости авто для сервиса оценки. Построим SHAP графики, исследуем зависимости и featrure importances. //// дополнить
# Стек:
pandas, catboost, scikit-learn, seaborn, numpy, matplotlib

1. Импорт библиотек и конфигурация проекта

In [7]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": True,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    # Категориальные признаки (определены в ходе предварительного анализа данных)
    "CAT_FEATURES": ["brand", "model", "transmission", "drive_type"]
}

In [71]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

2. Загрузка и первичный осмотр данных

In [ ]:
dates_only = pd.read_csv('../data/raw_dataset.csv', usecols=['Дата размещения объявления'])
print("Самая поздняя дата в файле:", dates_only['Дата размещения объявления'].max())

Заметим, что самая поздняя дата объявления - 2025-07-14. Так как файл слишком большой (5гб), отберём только объявления, размещенные с 2025-01-14 по 2025-07-14: это позволит не только облегчить вычисления, но и сделает будущую модель лучше, ведь она будет обучена на относительно "свежих" данных (учитываем инфляцию и актуальность цен).

In [9]:
date = 'Дата размещения объявления'
chunks = []
for chunk in pd.read_csv('../data/raw_dataset.csv', chunksize = 100000, low_memory=False):
    filtered = chunk[chunk[date].between('2025-01-14','2025-07-14')]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index = True)
df.shape

(585855, 58)

585855 строк - оптимальное значенение для обучения модели. Больше брать смысла нет, так как качество модели растет логарифмически по отношению к объему данных. Для начала проверим, есть ли в нашем датасете информация о спецтехнике.

In [ ]:
trucks_count = df['Тип техники'].notna().sum()
print(f"Найдено коммерческой техники/спецтехники: {trucks_count} шт.")

Найдено коммерческой техники/спецтехники: 0 шт.


Отлично! Никаких грузовиков, тягачей и кранов в нашем полном датасете нет - можно спокойно работать с нашим последующим сэмплом в 100к, не боясь что мы удалим важные столбцы для нелегковых автомобилей.
Все эксперименты будем проводить на DEV_MODE = True, чтобы работать с 100тыс. строк. В конце работы в CONFIG поменяем значение на False -> финальный запуск на всем объеме (585к строк)

In [ ]:
if CONFIG['DEV_MODE']:
    df = df.sample(n=100000, random_state=42)
    print("Режим разработки (100к строк). Всё будет летать!")
else:
    df = df.copy()
    print("Финальный режим (585к строк). Обучаем итоговую модель.")

Режим разработки (100к строк). Всё будет летать!


In [11]:
df.info

<bound method DataFrame.info of                Название машины     Год  \
242569               Лада 2112  2004.0   
474584     Toyota Corolla Axio  2009.0   
107914          Hyundai Sonata  2003.0   
108130          Hyundai Matrix  2005.0   
408139             Peugeot 308  2011.0   
...                        ...     ...   
485268          Toyota Corolla  2000.0   
124399            Hyundai Getz  2008.0   
226529             Лада Гранта  2015.0   
532695  Toyota Corolla Fielder  2007.0   
404789             Peugeot 308  2008.0   

                                                   Ссылка  \
242569  https://auto.drom.ru/berezanskaya/lada/2112/86...   
474584  https://auto.drom.ru/blagoveshchensk/toyota/co...   
107914  https://auto.drom.ru/pavlovskiy-posad/hyundai/...   
108130  https://auto.drom.ru/belorechensk/hyundai/matr...   
408139  https://auto.drom.ru/tambov/peugeot/308/915266...   
...                                                   ...   
485268  https://auto.drom.ru/chita/t

In [12]:
df.head(5)

,Название машины,Год,Ссылка,Дата размещения объявления,Цена,Кол-во просмотров,Скрыто,Объем двигателя,Тип двигателя,Мощность,...,Объем ковша,Длина стрелы,Грузоподъемность стрелы,Высота вышки,Состояние,Страна производства,Высота подъема,Ошибка_ст,Ошибка_знач,Пропуски в данных
242569,Лада 2112,2004.0,https://auto.drom.ru/berezanskaya/lada/2112/86...,2025-04-13,95000.0,350.0,0.0,1.6,бензин,89.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
474584,Toyota Corolla Axio,2009.0,https://auto.drom.ru/blagoveshchensk/toyota/co...,2025-02-19,890000.0,153.0,0.0,1.8,бензин,136.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"15,","0.0,","20,"
107914,Hyundai Sonata,2003.0,https://auto.drom.ru/pavlovskiy-posad/hyundai/...,2025-05-13,520000.0,70.0,0.0,2.4,бензин,138.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"15,","0.0,","20,"
108130,Hyundai Matrix,2005.0,https://auto.drom.ru/belorechensk/hyundai/matr...,2025-01-25,460000.0,50.0,0.0,1.8,бензин,122.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,"
408139,Peugeot 308,2011.0,https://auto.drom.ru/tambov/peugeot/308/915266...,2025-04-20,495000.0,33.0,0.0,1.6,бензин,120.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
(df.isnull().mean() * 100).round(2)

Название машины                 0.00
Год                             0.00
Ссылка                          0.00
Дата размещения объявления      0.00
Цена                            0.00
Кол-во просмотров               0.00
Скрыто                          0.00
Объем двигателя                 0.01
Тип двигателя                   0.00
Мощность                        0.01
Коробка передач                 0.00
Привод                          0.01
Пробег                          0.77
Руль                            0.04
Поколение                       0.02
Рестайлинг                      0.02
Цвет                            0.37
Комплектация                    0.23
Владелец                        0.00
Особые отметки                 93.50
Тип кузова                      0.42
VIN                            99.08
Проверено                     100.00
Номер кузова                   99.99
Метка                           0.00
Город                           0.00
Регион                          0.00
М

Заметим, что такие данные как высота подъема, объем ковша, высота седла, тип кабины и др. на 100% отсутствуют. Это связано с тем, что в данном датасете только легковые автомобили. Сможем смело удалять эти столбцы

3. Очистка данных

In [83]:
(df.isnull().mean() * 100).round(2)

Название машины                 0.00
Год                             0.00
Ссылка                          0.00
Дата размещения объявления      0.00
Цена                            0.00
Кол-во просмотров               0.00
Скрыто                          0.00
Объем двигателя                 0.01
Тип двигателя                   0.00
Мощность                        0.01
Коробка передач                 0.00
Привод                          0.01
Пробег                          0.77
Руль                            0.04
Поколение                       0.02
Рестайлинг                      0.02
Цвет                            0.37
Комплектация                    0.23
Владелец                        0.00
Особые отметки                 93.50
Тип кузова                      0.42
VIN                            99.08
Проверено                     100.00
Номер кузова                   99.99
Метка                           0.00
Город                           0.00
Регион                          0.00
М

In [36]:
df['Есть особые отметки'] = df['Особые отметки'].notna().astype(int)

In [37]:
# Задаем порог: если пропусков больше, чем 50% - смело удаляем столбец
threshold = len(df) * 0.5  
df_cleaned = df.dropna(thresh=threshold, axis=1)
print('Было колонок: ', df.shape[1])
print('Стало колонок: ', df_cleaned.shape[1])

Было колонок:  59
Стало колонок:  27


In [84]:
# Заполним медианой столбцы с числовыми признаками

num_cols = ['Мощность', 'Пробег', 'Объем двигателя']
for col in num_cols:
    global_median = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(
        df_cleaned.groupby('Название машины')[col].transform('median')
        )

In [85]:
# Заполним модой столбцы с категориальными признаками

cat_cols = ['Привод', 'Руль', 'Тип кузова', 'Тип двигателя', 'Владельцы', 'Коробка передач']
for col in cat_cols:
    global_mode = df_cleaned[col].mode()[0]
    df_cleaned[col] = df_cleaned.groupby('Название машины')[col].transform(
        lambda x: x.fillna(x.mode().get(0, global_mode))
        )

In [86]:
# Особенный случай - поколение зависит и от модели, и от года выпуска

df_cleaned['Поколение'] = df_cleaned.groupby(['Название машины', 'Год'])['Поколение'].transform(
    lambda x: x.fillna(x.mode().get(0, df_cleaned['Поколение'].mode()[0]))
    )

In [88]:
# Убираем ненужны столбцы
cols_to_drop = ['Дата размещения объявления', 'Кол-во просмотров', 'Скрыто', 'Ссылка', 'Владелец']
df_cleaned.drop(columns=cols_to_drop, inplace=True)

In [93]:
# Переименовываем "Метку" в понятную "Марку"
df_cleaned.rename(columns={'Метка': 'Марка'}, inplace=True)

In [95]:
# Проверяем
display(df_cleaned.isna().sum().sum())
df_cleaned.columns
# Все ок!

np.int64(0)

Index(['Название машины', 'Год', 'Цена', 'Объем двигателя', 'Тип двигателя',
       'Мощность', 'Коробка передач', 'Привод', 'Пробег', 'Руль', 'Поколение',
       'Рестайлинг', 'Цвет', 'Комплектация', 'Тип кузова', 'Марка', 'Город',
       'Регион', 'Макро-регион', 'Владельцы', 'Есть особые отметки'],
      dtype='object')

4. Исследовательский анализ данных (EDA)

5. Сохранение результата